In [1]:
%load_ext autoreload
%autoreload 2

MODEL = "gemini-3.1-flash-lite"


In [2]:
from sklearnex import patch_sklearn
patch_sklearn()

Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


In [3]:
from sulku.dataset import load_paired_dataset
from sulku.utils import sentencize, strip_markdown

ds = load_paired_dataset(MODEL)
print(f"Loaded {len(ds)} paired dataset items for model: {MODEL}")

Loaded 500 paired dataset items for model: gemini-3.1-flash-lite


In [4]:
human, machine = ds.sample(1)[0]
sentences = sentencize(strip_markdown(str(machine)))

sentences


['Sosterin alueella kaksi uutta koronavirustartuntaa – altistumisia bussilinjalla\nItä-Savon sairaanhoitopiirin kuntayhtymä Sosteri on vahvistanut tiistaina kaksi uutta koronavirustartuntaa.',
 'Toinen tapauksista on peräisin alueen ulkopuolelta, ja se on mahdollisesti altistanut matkustajia viime perjantaina Savonlinnan ja Varkauden välillä liikennöineessä Soisalon Liikenteen bussissa.',
 'Sosteri kehottaa kaikkia kyseisessä vuorossa matkustaneita hakeutumaan testiin oireista riippumatta.',
 'Toinen tartunta liittyy aiempaan tautiryppääseen, joka käsittää nyt viisi tapausta.',
 'Lisäksi 12 lasta on asetettu karanteeniin toisessa kunnassa tapahtuneen altistumisen vuoksi.',
 'Tällä hetkellä Sosterin alueella on karanteenissa noin 60 henkilöä.',
 'Alueen epidemiatilanne pysyy kuitenkin perustasolla.']

In [5]:
# Create a fastext training file with the sentences and their corresponding labels
    
# Perform document-level train/validation split (e.g., 80% train, 20% val)
train_size = int(len(ds) * 0.8)
train_pairs = ds[:train_size]
val_pairs = ds[train_size:]

# Separate into individual DatasetItem lists
train_human = [pair.source for pair in train_pairs]
train_synthetic = [pair.synthetic for pair in train_pairs]

print(f"Training dataset: {len(train_pairs)} items.")
print(f"Validation dataset: {len(val_pairs)} items.")

Training dataset: 400 items.
Validation dataset: 100 items.


In [ ]:
# Generate dataset for fastText training
from tempfile import mkstemp
from sulku.constants import LABEL_AI, LABEL_HUMAN

from sulku.dataset import generate_fasttext_sentence_data

def lines(file_path):
    # Count the number of lines in the training file
    with open(file_path, 'r', encoding='utf-8') as f:
        num_lines = sum(1 for line in f)
    return num_lines

_, training_file = mkstemp(suffix=".txt")

generate_fasttext_sentence_data(train_human, label=LABEL_HUMAN, output_path=training_file, mode="a")
generate_fasttext_sentence_data(train_synthetic, label=LABEL_AI, output_path=training_file, mode="a")

# Build a validation file and test the trained fastText model.
_, validation_file = mkstemp(suffix=".txt")

val_human = [pair.source for pair in val_pairs]
val_synthetic = [pair.synthetic for pair in val_pairs]

generate_fasttext_sentence_data(val_human, label=LABEL_HUMAN, output_path=validation_file, mode="a")
generate_fasttext_sentence_data(val_synthetic, label=LABEL_AI, output_path=validation_file, mode="a")


print(f"Training file: {training_file} ({lines(training_file)} lines)")
print(f"Validation file: {validation_file} ({lines(validation_file)} lines)")

Training file: /tmp/tmp8oxw1sir.txt (17522 lines)
Validation file: /tmp/tmp_c79kl4n.txt (4286 lines)


LLMs heavily favor specific prefixes, suffixes, and structural patterns (like ending words with certain morphemes).

Force character-level n-grams by setting minn=3 and maxn=6. This forces fastText to look inside the words.

In [7]:
import fasttext

# Set limits to constrain the tuning process
MAX_DURATION_SECONDS = 300 # 5 minutes
TARGET_MODEL_SIZE = "10M"  # Useful for edge deployment constraints

print("Starting autotune search...")
# The model will train iteratively, testing different combinations against the validation file
best_model = fasttext.train_supervised(
    input=training_file,
    autotuneValidationFile=validation_file,
    autotuneDuration=MAX_DURATION_SECONDS, 
#    autotuneModelSize=TARGET_MODEL_SIZE 
    # minn=3,     # Minimum length of char ngram
    # maxn=6      # Maximum length of char ngram
)

# Test the final, optimized model
n_examples, precision, recall = best_model.test(validation_file)
print(f"Optimized Precision: {precision:.4f}")
print(f"Optimized Recall: {recall:.4f}")

# Extract the chosen hyperparameters to see what the tool selected
print(f"Best Learning Rate (lr): {best_model.lr}")
print(f"Best Epochs (epoch): {best_model.epoch}")
print(f"Best Word N-grams (wordNgrams): {best_model.wordNgrams}")

Starting autotune search...


Progress: 100.0% Trials:   24 Best score:  0.837144 ETA:   0h 0m 0s
Training again with best arguments
Read 0M words
Number of words:  70281
Number of labels: 2
Progress: 100.0% words/sec/thread:   33296 lr:  0.000000 avg.loss:  0.232576 ETA:   0h 0m 0s 44.6% words/sec/thread:   33394 lr:  0.016494 avg.loss:  0.317245 ETA:   0h 0m18s% words/sec/thread:   33496 lr:  0.007468 avg.loss:  0.264177 ETA:   0h 0m 8s 0.007046 avg.loss:  0.263107 ETA:   0h 0m 7s


Optimized Precision: 0.8369
Optimized Recall: 0.8369
Best Learning Rate (lr): 0.02977478480779768
Best Epochs (epoch): 62
Best Word N-grams (wordNgrams): 3


In [ ]:
from pathlib import Path
output_file = f"/app/data/models/{MODEL}.ftz"
Path(output_file).parent.mkdir(parents=True, exist_ok=True)

best_model.quantize(training_file, retrain=True, verbose=True)
best_model.save_model(output_file)

n, p, r = best_model.test(validation_file)
print(f"Quantized Precision: {p:.4f}")
print(f"File size: {Path(output_file).stat().st_size / (1024 * 1024):.2f} MB")

Quantized Precision: 0.8299
